In [ ]:
import os
import kagglehub
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

path = kagglehub.dataset_download("emmarex/plantdisease")

# Resize and normalise every image the same way (128x128)
tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

healthy_imgs, diseased_imgs = [], []
max_per_class = 400

for root, _, files in os.walk(path):
    for f in sorted(files):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            full_path = os.path.join(root, f)
            folder = os.path.basename(root).lower()
            if "healthy" in folder and len(healthy_imgs) < max_per_class:
                healthy_imgs.append(tfm(Image.open(full_path).convert("RGB")))
            elif "healthy" not in folder and len(diseased_imgs) < max_per_class:
                diseased_imgs.append(tfm(Image.open(full_path).convert("RGB")))
        if len(healthy_imgs) >= max_per_class and len(diseased_imgs) >= max_per_class:
            break
    if len(healthy_imgs) >= max_per_class and len(diseased_imgs) >= max_per_class:
        break

X = torch.stack(healthy_imgs + diseased_imgs)
y = torch.tensor([0] * len(healthy_imgs) + [1] * len(diseased_imgs), dtype=torch.long)

loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
print(f"Loaded {len(X)} images | Tensor shape: {tuple(X.shape)} | Batches: {len(loader)}")

Using device: cuda
Using Colab cache for faster access to the 'plantdisease' dataset.
Loaded 800 images | Tensor shape: (800, 3, 128, 128) | Batches: 25


In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.LazyLinear(2),  # 0 = healthy, 1 = diseased
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
for epoch in range(1, 11):
    model.train()
    total_loss, correct = 0.0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        correct += (out.argmax(dim=1) == yb).sum().item()

    print(f"Epoch {epoch:2d} | Loss: {total_loss / len(X):.4f} | Acc: {correct / len(X):.2%}")

Epoch  1 | Loss: 0.5643 | Acc: 72.50%
Epoch  2 | Loss: 0.2666 | Acc: 89.50%
Epoch  3 | Loss: 0.1675 | Acc: 94.12%
Epoch  4 | Loss: 0.1226 | Acc: 95.00%
Epoch  5 | Loss: 0.0762 | Acc: 98.12%
Epoch  6 | Loss: 0.0420 | Acc: 98.75%
Epoch  7 | Loss: 0.0238 | Acc: 99.50%
Epoch  8 | Loss: 0.0262 | Acc: 99.25%
Epoch  9 | Loss: 0.0197 | Acc: 99.75%
Epoch 10 | Loss: 0.0097 | Acc: 100.00%
